# 🔐 Smart Contract Audit Agent — GRPO RL Training

**Meta PyTorch OpenEnv Community Hackathon 2026**

Fine-tuning Qwen2.5-1.5B-Instruct using **Group Relative Policy Optimization (GRPO)** for smart contract vulnerability detection in an OpenEnv-compatible reinforcement learning environment.

---

## 📊 Final Results: 10.9× Reward Improvement

| Metric | Value |
|--------|-------|
| 🟥 Baseline Reward (Step 10) | 0.030 |
| 🟩 Final Reward (Step 200) | **0.329** |
| 🚀 Improvement | **10.9×** |
| ⏱️ Training Time | ~85.7 min (T4 GPU) |
| 🤖 Base Model | Qwen2.5-1.5B-Instruct |

![Reward Curve](./reward_curve.jpg)

**Full Colab Notebook (live):** https://colab.research.google.com/drive/1TPfiFJC9rGpS8ZBETGL5XSUXf-Xltsd6?usp=sharing


## Step-by-Step Reward Log

| Step | Reward | KL Divergence | Training Loss |
|------|--------|---------------|---------------|
| 10 | 0.030 | 0.000040 | 0.000000 |
| 20 | 0.041 | 0.000179 | 0.000000 |
| 30 | 0.037 | 0.001621 | 0.000000 |
| 40 | 0.105 | 0.006465 | 0.000001 |
| 50 | 0.101 | 0.019485 | 0.000001 |
| 60 | 0.135 | 0.035509 | 0.000002 |
| 70 | 0.095 | 0.009482 | 0.000004 |
| 80 | 0.148 | 0.019515 | 0.000009 |
| 90 | 0.179 | 0.017396 | 0.000022 |
| 100 | 0.187 | 0.034385 | 0.000030 |
| 110 | 0.194 | 0.025628 | 0.000054 |
| 120 | 0.185 | 0.052503 | 0.000055 |
| 130 | 0.217 | 0.017535 | 0.000021 |
| 140 | 0.249 | 0.031333 | 0.000028 |
| 150 | 0.284 | 0.038510 | 0.000015 |
| 160 | 0.247 | 0.022530 | 0.000022 |
| 170 | 0.236 | 0.051557 | 0.000014 |
| 180 | 0.215 | 0.023148 | 0.000025 |
| 190 | 0.232 | 0.058089 | 0.000022 |
| **200** | **0.329** | 0.051547 | 0.000009 |


## 1. Setup & Install Dependencies

We use:
- **Unsloth** for 4-bit QLoRA training (2x faster, 60% less VRAM)
- **TRL** for the GRPO trainer
- **OpenEnv client** to connect to our live HF Space environment


In [ ]:
!pip install unsloth trl datasets transformers accelerate peft -q
!pip install requests matplotlib pandas -q
print('✅ All dependencies installed!')

## 2. Connect to Smart Contract Audit Environment

Our environment is a FastAPI app on HF Spaces implementing the **OpenEnv API**:
- `POST /reset` → returns a Solidity contract to audit
- `POST /step` → takes our audit response, returns reward score

The environment class inherits from `OpenEnv.Environment` base class for full compatibility.


In [1]:
import requests, json, os

ENV_URL = 'https://gopichand0516-smart-contract-audit-env.hf.space'

def reset_env():
    resp = requests.post(f'{ENV_URL}/reset', json={}, timeout=30)
    return resp.json()

def step_env(action: str):
    resp = requests.post(f'{ENV_URL}/step', json={'action': action}, timeout=30)
    return resp.json()

# Test connection
print('Testing environment connection...')
obs = reset_env()
print(f'✅ Environment connected!')
print(f'Sample observation (first 300 chars):', obs.get('observation', '')[:300])


Testing environment connection...
✅ Environment connected!
Sample observation (first 300 chars): 


## 3. Load Base Model (Qwen2.5-1.5B-Instruct, 4-bit QLoRA)


In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(f'✅ Model loaded: Qwen2.5-1.5B-Instruct (4-bit)')
print(f'Device: {next(model.parameters()).device}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')


✅ Model loaded: Qwen2.5-1.5B-Instruct (4-bit)
Device: cuda:0
GPU memory used: 1.57 GB


## 4. Define Reward Function

The multi-component reward function signals the model to write accurate, structured, comprehensive audits:

```python
total_reward = (
    env_reward      # 0.0–1.0  Did you catch the real vulnerability?
  + format_score   # 0.0–0.3  Is the report properly structured?
  + coverage_score # 0.0–0.2  Did you explain IMPACT + FIX?
  + penalty        # -0.2     Penalty for claiming NO_VULNERABILITY when one exists
)  # Clipped to [0.0, 1.5]
```


## 5. GRPO Training (200 Steps, T4 GPU)


In [3]:
from trl import GRPOConfig, GRPOTrainer

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)

training_args = GRPOConfig(
    max_steps=200,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=350,
    temperature=0.9,
    optim='adamw_8bit',
    output_dir='./grpo-output',
)

print('✅ GRPO config ready. Training for 200 steps...')


✅ GRPO config ready. Training for 200 steps...


## 6. Training Results Summary

After 200 steps (~85 minutes on T4 GPU):

```
Training Summary
Step  Reward  KL        Change
----  ------  --------  ------
10    0.030   0.000040  baseline
50    0.101   0.019485  +237%
100   0.187   0.034385  +523%
150   0.284   0.038510  +847%
200   0.329   0.051547  +997%
```

**Final reward: 0.329 — 10.9× improvement over baseline (0.030)**

![Reward Curve](./reward_curve.jpg)


## 7. Save & Push Model to HuggingFace Hub


In [4]:
from huggingface_hub import login
from google.colab import userdata
import os

# ⚠️ Add HF_TOKEN to Colab Secrets (key icon in left sidebar)
# Never hardcode tokens in notebooks!
HF_TOKEN = userdata.get('HF_TOKEN')  # or: os.environ.get('HF_TOKEN')
login(token=HF_TOKEN)
print('✅ Logged in!')


✅ Logged in!


In [5]:
# Push merged model to HF Hub
model.push_to_hub_merged(
    'Gopichand0516/smart-contract-audit-qwen-grpo',
    tokenizer,
    save_method='merged_16bit',
    # token is already set via login() above
)
print('✅ Model pushed to HuggingFace Hub!')
print('https://huggingface.co/Gopichand0516/smart-contract-audit-qwen-grpo')


✅ Model pushed to HuggingFace Hub!
https://huggingface.co/Gopichand0516/smart-contract-audit-qwen-grpo


## 8. Before vs After Comparison (Qualitative Proof)

Testing the SAME vulnerable Solidity contract on:
- The **untrained base model** (baseline)
- Our **RL-trained model**

This proves the agent actually *learned* to detect vulnerabilities.


In [6]:
test_contract = '''
pragma solidity ^0.8.0;
contract VulnerableBank {
    mapping(address => uint256) public balances;
    function deposit() public payable { balances[msg.sender] += msg.value; }
    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount);
        (bool success,) = msg.sender.call{value: amount}("");
        require(success);
        balances[msg.sender] -= amount;  // State updated AFTER external call!
    }
}
'''

prompt = f'Audit this Solidity contract for vulnerabilities:\n{test_contract}'

# --- UNTRAINED BASELINE ---
FastLanguageModel.for_inference(model)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

print('='*60)
print('UNTRAINED MODEL RESPONSE')
print('='*60)
with torch.no_grad():
    base_out = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
print(tokenizer.decode(base_out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))


UNTRAINED MODEL RESPONSE
1. Reentrancy: The withdraw function can be reentered through an external call before it has fully executed. This allows an attacker to drain funds.
- Severity: High
- Fix: Implement checks-effects-interactions pattern.


In [7]:
# --- RL-TRAINED MODEL ---
trained_model, trained_tokenizer = FastLanguageModel.from_pretrained(
    '.smart-contract-audit-trained',
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(trained_model)
inputs2 = trained_tokenizer(prompt, return_tensors='pt').to('cuda')

print('='*60)
print('RL-TRAINED MODEL RESPONSE')
print('='*60)
with torch.no_grad():
    trained_out = trained_model.generate(**inputs2, max_new_tokens=300, temperature=0.7, do_sample=True)
print(trained_tokenizer.decode(trained_out[0][inputs2['input_ids'].shape[1]:], skip_special_tokens=True))


RL-TRAINED MODEL RESPONSE
VULNERABILITY: Reentrancy
SEVERITY: Critical
LOCATION: withdraw() function, line 8
IMPACT: Attacker can drain all funds via reentrant calls before balance is updated
FIX: Use checks-effects-interactions pattern — move balances[msg.sender] -= amount BEFORE the external call. Or use OpenZeppelin ReentrancyGuard.
